# Foundations — Attention From Scratch

**Category:** Foundations · *Attention Is All You Need*, ResNet, BatchNorm, ViT, Mamba, SAM2.

**What this notebook does.** Implements scaled dot-product attention from scratch in NumPy on a tiny copy-task, then shows what happens when you remove the residual connection. The point isn't performance — it's mechanical intuition for why this substrate is everywhere.

**Runs on CPU in under a minute.** No GPU, no model downloads.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(7)

## 2. Scaled dot-product attention

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights


## 3. Toy copy task

Query a sequence of 6 tokens that each carry an embedding plus a positional cue. The 'correct' attention pattern is the identity — each query should attend to its own key. We construct K so that diagonal alignment is the right answer.

In [ ]:
T, d = 6, 8
tokens = np.random.randn(T, d)
Q = tokens.copy()
K = tokens.copy()  # query-key alignment by construction
V = np.eye(T)      # values are one-hot — easy to read off the attention pattern

out, attn = attention(Q, K, V)
print('Attention pattern (rows=queries, cols=keys):')
print(np.round(attn, 2))


In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(attn, cmap='viridis')
ax.set_title('Attention weights')
ax.set_xlabel('Key index'); ax.set_ylabel('Query index')
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()


Diagonal-dominant — every query found its matching key. Notice the *softness* of the diagonal. That softness is exactly what lets a Transformer trade off between exact lookup and statistical smoothing.

## 4. Residual connections — what they prevent

Simulate a 30-layer net where each layer is a noisy linear map. Compare: (a) with residuals, (b) without. The gradient signal collapses without skip connections.

In [ ]:
def forward(x, n_layers=30, residual=True):
    norms = [np.linalg.norm(x)]
    for _ in range(n_layers):
        W = np.random.randn(*x.shape) * 0.1
        delta = np.tanh(W * x)
        x = x + delta if residual else delta
        norms.append(np.linalg.norm(x))
    return norms

x0 = np.random.randn(64)
with_res = forward(x0.copy(), residual=True)
without_res = forward(x0.copy(), residual=False)

plt.figure(figsize=(6,3))
plt.plot(with_res, label='with residual')
plt.plot(without_res, label='without residual')
plt.xlabel('Layer'); plt.ylabel('Activation norm')
plt.title('Why skip connections matter')
plt.legend(); plt.tight_layout(); plt.show()


## My take

Attention is *learnable indexing*. That's the framing that has aged best. ResNet's residual is *don't make the model forget the input it just received*. BatchNorm is *don't let the layer's input distribution drift on you mid-training.* Three small mechanical safeguards are 80% of why deep networks train at all.

When I evaluate a new architecture in 2026, I ask the same three questions: where is the learnable index, where is the identity path, and where is the implicit normalization? If two of the three are missing, expect training instability.